<a href="https://colab.research.google.com/github/adljna/ProjectA-Group3-KematianAliKhamenei/blob/main/POS%20Tagging/Tribunnews/3_Tribunnews_POSTagging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **POS Tagging Berita Kematian Ali Khamenei - Detik**

Notebook ini berisi proses Part-of-Speech (POS) Tagging pada data artikel yang telah diproses sebelumnya. Langkah-langkah utama yang dilakukan dalam notebook ini meliputi:

*   **Instalasi & Setup**: Menginstal dan mengimpor pustaka Python yang diperlukan seperti `pandas`, `nltk`, dan `stanza` untuk pemrosesan bahasa alami (NLP) Indonesia.
*   **Load Data**: Mengunggah dan memuat file CSV (`analyzed_articles.csv`) yang berisi data artikel yang sudah melalui tahap pra-pemrosesan (seperti pembersihan teks, tokenisasi, penghapusan stopwords, stemming, dan analisis sentimen).
*   **POS Tagging**: Menerapkan POS tagging pada kolom `text_final` menggunakan model Stanza bahasa Indonesia untuk mengidentifikasi kategori gramatikal setiap kata (misalnya, kata benda, kata kerja, kata sifat).
*   **Analisis Frekuensi POS**: Menganalisis dan menampilkan frekuensi kata-kata teratas untuk setiap jenis POS yang relevan, memberikan wawasan tentang distribusi kata dalam teks.
*   **Ekspor Hasil**: Menyimpan DataFrame yang telah diperbarui, termasuk kolom `pos_tags` yang baru, ke dalam file CSV baru (`tagged_articles.csv`).

# **(1) Installation & Setup**

Ini adalah bagian untuk menginstal dan mengimpor semua pustaka Python yang diperlukan untuk analisis teks. Ini mencakup `pandas` untuk manipulasi data, `nltk` dan `stanza` untuk pemrosesan bahasa alami (NLP) Indonesia, `tqdm` untuk progress bar, `io` untuk input/output data, dan `google.colab.files` untuk mengunggah file. Model bahasa Indonesia untuk NLTK dan Stanza juga diunduh dan diinisialisasi di sini.

In [1]:
!pip install nlp-id transformers torch tqdm stanza

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import files
import io
import pandas as pd
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
import stanza
from collections import Counter, defaultdict
from tqdm import tqdm

tqdm.pandas()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


# **(2) Load Data**

Pada bagian ini, Anda akan mengunggah file CSV yang berisi data artikel. Kode akan membaca file yang diunggah ke dalam DataFrame `df_article` menggunakan pustaka `pandas` dan `io`.

In [3]:
from google.colab import files
uploaded = files.upload()

import io
import pandas as pd

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

  # Assuming the uploaded file is a CSV
  try:
    df_article = pd.read_csv(io.BytesIO(uploaded[fn]))
    print("Successfully loaded CSV into df_article")
  except pd.errors.ParserError:
      print(f"Error: Could not parse {fn} as a CSV. Please upload a valid CSV file.")
      df_article = pd.DataFrame() # Create an empty DataFrame if parsing fails
  except Exception as e:
    print(f"An unexpected error occurred: {e}")
    df_article = pd.DataFrame()

Saving analyzed_articles.csv to analyzed_articles.csv
User uploaded file "analyzed_articles.csv" with length 1778553 bytes
Successfully loaded CSV into df_article


In [4]:
print(df_article.columns)

Index(['url', 'title', 'content', 'content_cleaned_step2', 'text_cleaned',
       'tokens', 'tokens_no_stop', 'tokens_stemmed', 'text_final', 'polarity',
       'subjectivity', 'sentiment_label'],
      dtype='object')


In [5]:
df_article

,url,title,content,content_cleaned_step2,text_cleaned,tokens,tokens_no_stop,tokens_stemmed,text_final,polarity,subjectivity,sentiment_label
0,https://aceh.tribunnews.com/news/1018596/duka-...,"Duka dan Amarah di Pemakaman Komandan AL IRGC,...",SERAMBINEWS.COM - Ribuan warga memadati jalana...,Ribuan warga memadati jalanan Teheran pada Rab...,ribuan warga memadati jalanan teheran pada rab...,"['ribuan', 'warga', 'memadati', 'jalanan', 'te...","['ribuan', 'warga', 'memadati', 'jalanan', 'te...","['ribu', 'warga', 'padat', 'jalan', 'teheran',...",ribu warga padat jalan teheran rabu iring maka...,0.022267,0.501246,Netral
1,https://video.tribunnews.com/news/916067/media...,Media Iran Konfirmasi Kematian Ali Khamenei Be...,Download aplikasi berita TribunX di Play Store...,"Dikutip dari media resmi milik Iran, Press TV,...",dikutip dari media resmi milik iran press tv k...,"['dikutip', 'dari', 'media', 'resmi', 'milik',...","['dikutip', 'media', 'resmi', 'milik', 'iran',...","['kutip', 'media', 'resmi', 'milik', 'iran', '...",kutip media resmi milik iran press tv kantor b...,0.100152,0.337289,Positive
2,https://wartakota.tribunnews.com/news/883298/p...,"Pemimpin Tertinggi Iran Tewas, Ustaz Felix Sia...",Ringkasan Berita:\n\nPresiden AS Donald Trump ...,Presiden AS Donald Trump mengumumkan tewasnya ...,presiden as donald trump mengumumkan tewasnya ...,"['presiden', 'as', 'donald', 'trump', 'mengumu...","['presiden', 'as', 'donald', 'trump', 'mengumu...","['presiden', 'as', 'donald', 'trump', 'umum', ...",presiden as donald trump umum tewas pimpin tin...,-0.009600,0.389438,Netral
3,https://aceh.tribunnews.com/news/1018596/duka-...,"Duka dan Amarah di Pemakaman Komandan AL IRGC,...",SERAMBINEWS.COM - Ribuan warga memadati jalana...,Ribuan warga memadati jalanan Teheran pada Rab...,ribuan warga memadati jalanan teheran pada rab...,"['ribuan', 'warga', 'memadati', 'jalanan', 'te...","['ribuan', 'warga', 'memadati', 'jalanan', 'te...","['ribu', 'warga', 'padat', 'jalan', 'teheran',...",ribu warga padat jalan teheran rabu iring maka...,0.022267,0.501246,Netral
4,https://palu.tribunnews.com/news/180470/ditemu...,"Ditemukan di Bawah Reruntuhan, Ini Kronologi T...",TRIBUNPALU.COM - Jenazah Pemimpin Tertinggi Ir...,"Jenazah Pemimpin Tertinggi Iran, Ali Khamenei,...",jenazah pemimpin tertinggi iran ali khamenei d...,"['jenazah', 'pemimpin', 'tertinggi', 'iran', '...","['jenazah', 'pemimpin', 'tertinggi', 'iran', '...","['jenazah', 'pimpin', 'tinggi', 'iran', 'ali',...",jenazah pimpin tinggi iran ali khamenei temu p...,0.042853,0.440718,Netral
...,...,...,...,...,...,...,...,...,...,...,...,...
95,https://video.tribunnews.com/news/923072/rudal...,RUDAL IRAN HANTAM HERZLIYA! Warga Israel: Siri...,Download aplikasi berita TribunX di Play Store...,"Warga bernama Shimon mengatakan, sirine pering...",warga bernama shimon mengatakan sirine peringa...,"['warga', 'bernama', 'shimon', 'mengatakan', '...","['warga', 'bernama', 'shimon', 'sirine', 'peri...","['warga', 'nama', 'shimon', 'sirine', 'ingat',...",warga nama shimon sirine ingat bunyi rudal ira...,0.450000,0.450000,Positive
96,https://kalteng.tribunnews.com/news/228967/kab...,"Kabar Terbaru Serangan AS-Israel, Iran Benarka...",Ringkasan Berita:\n\nIran mengkonfirmasi bahwa...,Iran mengkonfirmasi bahwa pemimpin tertinggi A...,iran mengkonfirmasi bahwa pemimpin tertinggi a...,"['iran', 'mengkonfirmasi', 'bahwa', 'pemimpin'...","['iran', 'mengkonfirmasi', 'pemimpin', 'tertin...","['iran', 'konfirmasi', 'pimpin', 'tinggi', 'ay...",iran konfirmasi pimpin tinggi ayatollah ali kh...,0.042076,0.386852,Netral
97,https://jakarta.tribunnews.com/news/431291/tru...,Trump Klaim Ayatollah Ali Khamenei Tewas dalam...,TRIBUNJAKARTA.COM - Di tengah suasana Ramadan ...,"Di tengah suasana Ramadan penuh khidmat, Indon...",di tengah suasana ramadan penuh khidmat indone...,"['di', 'tengah', 'suasana', 'ramadan', 'penuh'...","['suasana', 'ramadan', 'penuh', 'khidmat', 'in...","['suasana', 'ramadan', 'penuh', 'khidmat', '

# **(3) POS tagging**


Bagian ini menerapkan Part-of-Speech (POS) tagging pada kolom `text_final` dalam DataFrame `df_article`. Setiap kata dalam teks akan diberi label berdasarkan kelas katanya (misalnya, kata benda, kata kerja, kata sifat) menggunakan model `stanza` yang telah diinisialisasi. Hasilnya akan disimpan dalam kolom baru bernama `pos_tags`.

In [6]:
# Download the Indonesian Stanza model if not already downloaded
stanza.download('id')

# Initialize the Stanza pipeline for tokenization and POS tagging
stanza_nlp = stanza.Pipeline('id', processors='tokenize,pos')

# Create wrapper class to mimic the interface of the original pos_tagger
class StanzaPosTagger:
    def get_pos_tag(self, text):
        doc = stanza_nlp(text)
        pos_tags = []
        for sent in doc.sentences:
            for word in sent.words:
                pos_tags.append((word.text, word.pos))
        return pos_tags

pos_tagger = StanzaPosTagger()

INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: id (Indonesian) ...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/id/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: id (Indonesian):
| Processor | Package    |
--------------------------
| tokenize  | gsd        |
| mwt       | gsd        |
| pos       | gsd_charlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Done loading processors!


In [7]:
# Make sure the text column is string
df_article["text_final"] = df_article["text_final"].astype(str)

# Apply POS tagging
df_article["pos_tags"] = df_article["text_final"].apply(lambda x: pos_tagger.get_pos_tag(x))

In [8]:
# Flatten the list of lists of (word, tag) tuples into a single list of dictionaries
word_pos_list = []
for index, row in df_article.iterrows():
    if isinstance(row['pos_tags'], list):
        for word, pos_tag in row['pos_tags']:
            word_pos_list.append({'Word': word, 'POS Tag': pos_tag})

df_word_pos = pd.DataFrame(word_pos_list)
print("\nDataFrame Kata dan POS Tag dari Stanza:")
display(df_word_pos.head())

# Menghitung frekuensi tag dan kata unik per tag
tag_counts = df_word_pos['POS Tag'].value_counts()
unique_tokens_per_tag = df_word_pos.groupby('POS Tag')['Word'].nunique()

# Create summary DataFrame, ensuring index alignment for unique_tokens_per_tag
summary_df = pd.DataFrame({
    'Tag': tag_counts.index,
    'Count': tag_counts.values,
    'Unique Tokens': unique_tokens_per_tag.reindex(tag_counts.index)
}).reset_index(drop=True)
summary_df = summary_df.sort_values(by='Count', ascending=False)

print("\nRingkasan POS Tagging:")
display(summary_df)
print(f"Total kata yang di-tag: {summary_df['Count'].sum()}")

# Save summary to CSV
summary_df.to_csv("pos_tags_summary.csv", index=False)


DataFrame Kata dan POS Tag dari Stanza:


,Word,POS Tag
0,ribu,NOUN
1,warga,NOUN
2,padat,ADJ
3,jalan,NOUN
4,teheran,NOUN



Ringkasan POS Tagging:


,Tag,Count,Unique Tokens
0,NOUN,17592,1710
1,ADJ,1743,161
2,VERB,1000,122
3,PROPN,854,91
4,X,324,103
5,PART,204,3
6,ADP,84,17
7,PRON,73,7
8,ADV,68,15
9,NUM,60,10


Total kata yang di-tag: 22038


In [9]:
df_article[["text_final", "pos_tags"]].head(10)

,text_final,pos_tags
0,ribu warga padat jalan teheran rabu iring maka...,"[(ribu, NOUN), (warga, NOUN), (padat, ADJ), (j..."
1,kutip media resmi milik iran press tv kantor b...,"[(kutip, NOUN), (media, NOUN), (resmi, ADJ), (..."
2,presiden as donald trump umum tewas pimpin tin...,"[(presiden, PROPN), (as, PROPN), (donald, PROP..."
3,ribu warga padat jalan teheran rabu iring maka...,"[(ribu, NOUN), (warga, NOUN), (padat, ADJ), (j..."
4,jenazah pimpin tinggi iran ali khamenei temu p...,"[(jenazah, NOUN), (pimpin, NOUN), (tinggi, ADJ..."
5,wartakotalivecom pimpin tinggi iran ali khamen...,"[(wartakotalivecom, PROPN), (pimpin, VERB), (t..."
6,serambinewscom teheran pimpin tinggi iran moj...,"[(serambinewscom, NOUN), (teheran, NOUN), (pim..."
7,terang minggu pezeshkian mati ali khamenei jah...,"[(terang, NOUN), (minggu, NOUN), (pezeshkian, ..."
8,media perintah iran konfirmasi mati pimpin tin...,"[(media, NOUN), (perintah, NOUN), (iran, NOUN)..."
9,bawa acara berita kuasa tahan emosi baca umum ...,"[(bawa, VERB), (acara, NOUN), (berita, NOUN), ..."


# (4) POS Frequency Analysis

Kode ini menganalisis frekuensi kemunculan kata-kata berdasarkan tag POS-nya. Ini akan menghitung 5 kata teratas untuk setiap jenis POS yang relevan (seperti kata benda, kata kerja, kata sifat, dll.), memberikan gambaran tentang kata-kata yang paling sering digunakan dalam setiap kategori.

In [10]:
from collections import defaultdict, Counter

# --- Analisis Frekuensi POS (Part-of-Speech) ---

# Buat dictionary untuk menyimpan Counter untuk setiap tag POS
pos_counts = defaultdict(Counter)

# Iterasi setiap baris di DataFrame
# df_article['pos_tags'] berisi list of tuples, misal: [('lion', 'NN'), ('air', 'NN')]
for list_of_tags in df_article['pos_tags']:
    for word, tag in list_of_tags:
        pos_counts[tag][word] += 1

print("--- Frekuensi Top 5 Kata per Jenis POS ---")

# Tampilkan 5 kata paling umum untuk setiap tag POS yang relevan
# Stanza uses Universal Dependencies (UD) tags. Common ones include NOUN, VERB, ADJ, ADV, PROPN.
relevant_pos_tags = ['NOUN', 'VERB', 'ADJ', 'ADV', 'PROPN'] # Updated to use UD tags

for tag in relevant_pos_tags:
    if tag in pos_counts:
        print(f"\nTop 5 untuk tag '{tag}':")
        print(pos_counts[tag].most_common(5))


--- Frekuensi Top 5 Kata per Jenis POS ---

Top 5 untuk tag 'NOUN':
[('iran', 1080), ('khamenei', 702), ('serang', 541), ('pimpin', 379), ('israel', 314)]

Top 5 untuk tag 'VERB':
[('tewas', 208), ('tinggal', 67), ('bunuh', 56), ('sebut', 44), ('hancur', 33)]

Top 5 untuk tag 'ADJ':
[('tinggi', 224), ('militer', 198), ('nasional', 85), ('aman', 71), ('kuat', 66)]

Top 5 untuk tag 'ADV':
[('tentu', 21), ('sempat', 8), ('ka', 6), ('benarbenar', 6), ('turut', 5)]

Top 5 untuk tag 'PROPN':
[('amerika', 214), ('as', 207), ('serikat', 66), ('trump', 35), ('israel', 20)]


In [11]:
df_article

,url,title,content,content_cleaned_step2,text_cleaned,tokens,tokens_no_stop,tokens_stemmed,text_final,polarity,subjectivity,sentiment_label,pos_tags
0,https://aceh.tribunnews.com/news/1018596/duka-...,"Duka dan Amarah di Pemakaman Komandan AL IRGC,...",SERAMBINEWS.COM - Ribuan warga memadati jalana...,Ribuan warga memadati jalanan Teheran pada Rab...,ribuan warga memadati jalanan teheran pada rab...,"['ribuan', 'warga', 'memadati', 'jalanan', 'te...","['ribuan', 'warga', 'memadati', 'jalanan', 'te...","['ribu', 'warga', 'padat', 'jalan', 'teheran',...",ribu warga padat jalan teheran rabu iring maka...,0.022267,0.501246,Netral,"[(ribu, NOUN), (warga, NOUN), (padat, ADJ), (j..."
1,https://video.tribunnews.com/news/916067/media...,Media Iran Konfirmasi Kematian Ali Khamenei Be...,Download aplikasi berita TribunX di Play Store...,"Dikutip dari media resmi milik Iran, Press TV,...",dikutip dari media resmi milik iran press tv k...,"['dikutip', 'dari', 'media', 'resmi', 'milik',...","['dikutip', 'media', 'resmi', 'milik', 'iran',...","['kutip', 'media', 'resmi', 'milik', 'iran', '...",kutip media resmi milik iran press tv kantor b...,0.100152,0.337289,Positive,"[(kutip, NOUN), (media, NOUN), (resmi, ADJ), (..."
2,https://wartakota.tribunnews.com/news/883298/p...,"Pemimpin Tertinggi Iran Tewas, Ustaz Felix Sia...",Ringkasan Berita:\n\nPresiden AS Donald Trump ...,Presiden AS Donald Trump mengumumkan tewasnya ...,presiden as donald trump mengumumkan tewasnya ...,"['presiden', 'as', 'donald', 'trump', 'mengumu...","['presiden', 'as', 'donald', 'trump', 'mengumu...","['presiden', 'as', 'donald', 'trump', 'umum', ...",presiden as donald trump umum tewas pimpin tin...,-0.009600,0.389438,Netral,"[(presiden, PROPN), (as, PROPN), (donald, PROP..."
3,https://aceh.tribunnews.com/news/1018596/duka-...,"Duka dan Amarah di Pemakaman Komandan AL IRGC,...",SERAMBINEWS.COM - Ribuan warga memadati jalana...,Ribuan warga memadati jalanan Teheran pada Rab...,ribuan warga memadati jalanan teheran pada rab...,"['ribuan', 'warga', 'memadati', 'jalanan', 'te...","['ribuan', 'warga', 'memadati', 'jalanan', 'te...","['ribu', 'warga', 'padat', 'jalan', 'teheran',...",ribu warga padat jalan teheran rabu iring maka...,0.022267,0.501246,Netral,"[(ribu, NOUN), (warga, NOUN), (padat, ADJ), (j..."
4,https://palu.tribunnews.com/news/180470/ditemu...,"Ditemukan di Bawah Reruntuhan, Ini Kronologi T...",TRIBUNPALU.COM - Jenazah Pemimpin Tertinggi Ir...,"Jenazah Pemimpin Tertinggi Iran, Ali Khamenei,...",jenazah pemimpin tertinggi iran ali khamenei d...,"['jenazah', 'pemimpin', 'tertinggi', 'iran', '...","['jenazah', 'pemimpin', 'tertinggi', 'iran', '...","['jenazah', 'pimpin', 'tinggi', 'iran', 'ali',...",jenazah pimpin tinggi iran ali khamenei temu p...,0.042853,0.440718,Netral,"[(jenazah, NOUN), (pimpin, NOUN), (tinggi, ADJ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,https://video.tribunnews.com/news/923072/rudal...,RUDAL IRAN HANTAM HERZLIYA! Warga Israel: Siri...,Download aplikasi berita TribunX di Play Store...,"Warga bernama Shimon mengatakan, sirine pering...",warga bernama shimon mengatakan sirine peringa...,"['warga', 'bernama', 'shimon', 'mengatakan', '...","['warga', 'bernama', 'shimon', 'sirine', 'peri...","['warga', 'nama', 'shimon', 'sirine', 'ingat',...",warga nama shimon sirine ingat bunyi rudal ira...,0.450000,0.450000,Positive,"[(warga, NOUN), (nama, NOUN), (shimon, NOUN), ..."
96,https://kalteng.tribunnews.com/news/228967/kab...,"Kabar Terbaru Serangan AS-Israel, Iran Benarka...",Ringkasan Berita:\n\nIran mengkonfirmasi bahwa...,Iran mengkonfirmasi bahwa pemimpin tertinggi A...,iran mengkonfirmasi bahwa pemimpin tertinggi a...,"['iran', 'mengkonfirmasi', 'bahwa', 'pemimpin'...","['iran', 'mengkonfirmasi', 'pemimpin', 'tertin...","['iran', 'konfirmasi', 'pimpin', 'tinggi', 'ay...",iran konfirmasi pimpin tinggi ayatollah ali kh...,0.042076,0.386852,Netral,"[(iran, NOUN), (konfirmasi, NOUN), (pimpin, NO..."
97,https://jakarta.tribunne

# (5) Save Processed Data

Setelah semua langkah pemrosesan selesai, DataFrame yang telah diperbarui dengan tag POS akan disimpan ke dalam file CSV baru dengan nama `data/tagged_articles.csv`. Parameter `index=False` memastikan bahwa indeks DataFrame tidak ikut disimpan ke dalam file CSV.

In [12]:
# Save the DataFrame to a CSV file
df_article.to_csv('tagged_articles.csv', index=False)